# 04. Decomposed SIR Evaluation

**Purpose:** Compute the decomposed Semantic Irrelevance Ratio (SIR*) comparing GCR, DCA-Trie v1, and DCA-Trie v2.

**What SIR measures (revised, §3.4):** At each decoding step $t$, what fraction of paths in the valid token set are semantically irrelevant? Irrelevance is the disjunction of two independent failure modes:
- **Type irrelevance** ($\text{SIR}^*_{\text{type}}$): terminal entity type mismatches the question's answer type
- **Trajectory irrelevance** ($\text{SIR}^*_{\text{traj}}$): decomposed relevance score below $\tau_{ref}=0.3$

**Reference:** Chapter 3, §3.4. Equations (3.5)-(3.11).

Key formulas:
```
irrel_type(p, q) = 1[type(e_h) ∉ T(q, h)]                    (Eq. 3.5)
irrel_traj(p, q, y_{<t}) = 1[score(p | q, y_{<t}) < τ_ref]  (Eq. 3.6)
irrel*(p, q, y_{<t}) = irrel_type ∨ irrel_traj               (Eq. 3.7)
SIR*_type(q,t) = count(irrel_type=1) / |P(q,t)|             (Eq. 3.8)
SIR*_traj(q,t) = count(irrel_traj=1) / |P(q,t)|             (Eq. 3.9)
SIR*(q,t) = count(irrel*=1) / |P(q,t)|                      (Eq. 3.10)
```

In [ ]:
# @title 1. Setup
import sys
import os
import json
import re
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict
from tqdm import tqdm

!pip install -q transformers==4.44.2 accelerate>=0.30.1 tiktoken>=0.7.0 datasets>=2.19.2 marisa-trie>=1.2.0 scikit-learn>=1.5.0 sentencepiece>=0.2.0 protobuf>=5.27.1 networkx>=3.0

if not os.path.exists("graph-constrained-reasoning"):
    !git clone https://github.com/Adjanour/graph-constrained-reasoning-dca

%cd graph-constrained-reasoning
sys.path.insert(0, '.')

from sentence_transformers import SentenceTransformer
from datasets import load_dataset
import src.utils as utils

In [ ]:
# @title 2. Configuration
# Fixed reference threshold (§3.4.2)
TAU_REF = 0.3

# Sentence encoder
ENCODER_NAME = "all-MiniLM-L6-v2"
ENCODER_DEVICE = "cpu"

# Dataset
DATA_PATH = "rmanluo"
DATASET = "RoG-webqsp"
SPLIT = "test"
INDEX_LEN = 2

# SAMPLING
MAX_SAMPLES = 100

# Point to each system's predictions
MODEL_SHORT = "GCR-Meta-Llama-3.1-8B-Instruct"

EXPERIMENTS = {
    "GCR": {
        "path": f"results/GenPaths/{DATASET}/{MODEL_SHORT}/{SPLIT}/zero-shot-group-beam-k5-index_len2/predictions.jsonl",
        "type": "gcr",
    },
    "DCA-Trie v1 (τ=0.25)": {
        "path": f"results/GenPaths/{DATASET}/{MODEL_SHORT}/{SPLIT}/zero-shot-group-beam-k5-index_len2-DCAv1-tau0.25-all-MiniLM-L6-v2/predictions.jsonl",
        "type": "dcav1",
    },
    "DCA-Trie v2 (τ=0.25)": {
        "path": f"results/GenPaths/{DATASET}/{MODEL_SHORT}/{SPLIT}/zero-shot-greedy-k5-index_len2-DCAv2-tau0.25-eps0.1-all-MiniLM-L6-v2/predictions.jsonl",
        "type": "dcav2",
    },
}

print(f"Reference threshold τ_ref: {TAU_REF}")
print(f"Encoder: {ENCODER_NAME}")
print(f"Dataset: {DATASET}")
for name, cfg in EXPERIMENTS.items():
    exists = os.path.exists(cfg["path"])
    print(f"  {name}: {'found' if exists else 'not found'} at {cfg['path']}")

In [ ]:
# @title 3. Type Oracle (reused from notebooks 02/03)

QUESTION_TYPE_PATTERNS = [
    (r"who\b", "person"),
    (r"what.*nationality", "country"),
    (r"what.*country", "country"),
    (r"what.*language", "language"),
    (r"what.*film|movie", "film"),
    (r"what.*award", "award"),
    (r"what.*occupation|job|profession", "occupation"),
    (r"where\b", "location"),
    (r"when\b", "date"),
    (r"which (country|city|state|place|location)", "location"),
    (r"which (person|people|actor|director|president)", "person"),
    (r"which (film|movie)", "film"),
    (r"which (language)", "language"),
    (r"which (award|prize|honor)", "award"),
]

RELATION_DOMAIN_MAP = {
    "people": "person", "person": "person", "location": "location",
    "film": "film", "tv": "tv_program", "award": "award",
    "education": "educational_institution", "organization": "organization",
    "sports": "sports_team", "language": "language",
    "base": "other", "common": "other",
}


class TypeOracle:
    """Type compatibility oracle for SIR type irrelevance computation."""

    def __init__(self, graph_triples):
        self.entity_types = self._build_type_index(graph_triples)

    @staticmethod
    def _domain_from_relation(rel):
        domain = rel.split(".")[0] if "." in rel else rel
        return RELATION_DOMAIN_MAP.get(domain, "other")

    def _build_type_index(self, triples):
        type_map = defaultdict(set)
        for h, r, t in triples:
            type_map[h].add(self._domain_from_relation(r))
            type_map[t].add("other")
        return type_map

    def get_terminal_types(self, question):
        q = question.lower()
        for pattern, etype in QUESTION_TYPE_PATTERNS:
            if re.search(pattern, q):
                return {etype}
        return {"person", "location", "organization", "date", "language", "film", "award", "other"}

    def is_type_compatible(self, entity, expected_types):
        if entity not in self.entity_types:
            return False
        entity_t = self.entity_types[entity]
        if "other" in expected_types:
            return True
        return len(entity_t & expected_types) > 0

In [ ]:
# @title 4. Load Encoder and Dataset
print("Loading encoder...")
encoder = SentenceTransformer(ENCODER_NAME, device=ENCODER_DEVICE)

print("Loading dataset...")
dataset = load_dataset(f"{DATA_PATH}/{DATASET}", split=SPLIT)
if MAX_SAMPLES is not None:
    dataset = dataset.select(range(min(MAX_SAMPLES, len(dataset))))
data_by_id = {d["id"]: d for d in dataset}
print(f"Loaded {len(data_by_id)} questions")

In [ ]:
# @title 5. Decomposed SIR Computation (§3.4.2–3.4.3)

def compute_decomposed_sir(predictions_path, encoder, data_by_id,
                           tau_ref, index_len=2, system_type="gcr"):
    """Compute decomposed SIR metrics for a single experiment.

    Returns:
      SIR*_type: fraction of paths with terminal entity type mismatch (Eq. 3.8)
      SIR*_traj: fraction of paths with score < τ_ref (Eq. 3.9)
      SIR*:     combined (disjunction, Eq. 3.10)
      Per-hop versions of all three.
    """
    if not os.path.exists(predictions_path):
        print(f"  File not found: {predictions_path}")
        return None

    with open(predictions_path, "r") as f:
        predictions = [json.loads(l) for l in f]

    hop_sir_type = defaultdict(list)
    hop_sir_traj = defaultdict(list)
    hop_sir_combined = defaultdict(list)
    hop_trie_sizes = defaultdict(list)

    for pred in tqdm(predictions, desc="Computing SIR"):
        qid = pred["id"]
        if qid not in data_by_id:
            continue
        data = data_by_id[qid]
        question = pred["question"]
        entities = data["q_entity"]

        g = utils.build_graph(data["graph"])
        if "paths" in data:
            paths_list = data["paths"]
        else:
            paths_list = utils.dfs(g, entities, index_len)

        if len(paths_list) == 0:
            continue

        path_strs = [utils.path_to_string(p) for p in paths_list]

        type_oracle = TypeOracle(data["graph"])
        t_terminal = type_oracle.get_terminal_types(question)

        u_q = encoder.encode(question, convert_to_numpy=True)

        path_embeddings = encoder.encode(path_strs, convert_to_numpy=True)

        paths_by_hop = defaultdict(list)
        strs_by_hop = defaultdict(list)
        emb_by_hop = defaultdict(list)
        for idx, p in enumerate(paths_list):
            h = len(p)
            paths_by_hop[h].append(p)
            strs_by_hop[h].append(path_strs[idx])
            emb_by_hop[h].append(path_embeddings[idx])

        for step in range(1, index_len + 1):
            if step not in paths_by_hop:
                continue

            hop_paths = paths_by_hop[step]
            hop_strs = strs_by_hop[step]
            hop_embs = np.array(emb_by_hop[step])
            n_total = len(hop_paths)

            type_irrelevant = 0
            for p in hop_paths:
                e_term = p[-1][2]
                if not type_oracle.is_type_compatible(e_term, t_terminal):
                    type_irrelevant += 1

            traj_scores = cosine_similarity(u_q.reshape(1, -1), hop_embs)[0]
            traj_irrelevant = int(np.sum(traj_scores < tau_ref))

            combined_irrelevant = 0
            for idx_p, p in enumerate(hop_paths):
                e_term = p[-1][2]
                type_bad = not type_oracle.is_type_compatible(e_term, t_terminal)
                traj_bad = traj_scores[idx_p] < tau_ref
                if type_bad or traj_bad:
                    combined_irrelevant += 1

            hop_sir_type[step].append(type_irrelevant / max(1, n_total))
            hop_sir_traj[step].append(traj_irrelevant / max(1, n_total))
            hop_sir_combined[step].append(combined_irrelevant / max(1, n_total))
            hop_trie_sizes[step].append(n_total)

    results = {}
    for step in sorted(hop_sir_type.keys()):
        results[f"sir_type_hop_{step}"] = float(np.mean(hop_sir_type[step]))
        results[f"sir_traj_hop_{step}"] = float(np.mean(hop_sir_traj[step]))
        results[f"sir_hop_{step}"] = float(np.mean(hop_sir_combined[step]))
        results[f"avg_trie_size_hop_{step}"] = float(np.mean(hop_trie_sizes[step]))

    all_type = [v for vals in hop_sir_type.values() for v in vals]
    all_traj = [v for vals in hop_sir_traj.values() for v in vals]
    all_combined = [v for vals in hop_sir_combined.values() for v in vals]

    results["corpus_sir_type"] = float(np.mean(all_type)) if all_type else 0.0
    results["corpus_sir_traj"] = float(np.mean(all_traj)) if all_traj else 0.0
    results["corpus_sir"] = float(np.mean(all_combined)) if all_combined else 0.0
    results["num_questions"] = len(predictions)
    results["num_observations"] = len(all_combined)

    return results

In [ ]:
# @title 6. Compute SIR for All Experiments
all_results = {}
for name, cfg in EXPERIMENTS.items():
    print(f"\nComputing SIR for: {name}")
    results = compute_decomposed_sir(
        cfg["path"], encoder, data_by_id, TAU_REF, INDEX_LEN, cfg["type"]
    )
    all_results[name] = results

In [ ]:
# @title 7. Results: Corpus-Level Decomposed SIR
print(f"\nDECOMPOSED SEMANTIC IRRELEVANCE RATIO (SIR*) - §3.4")
print(f"Reference threshold τ_ref = {TAU_REF}")
print(f"Dataset: {DATASET}\n")

header = f"{'System':<28} {'SIR*':<10} {'SIR*_type':<12} {'SIR*_traj':<12} {'Avg Trie':<10} {'N':<8}"
print(header)
print("-" * len(header))

for name, r in all_results.items():
    if r is None:
        print(f"{name:<28} {'N/A':<10}")
        continue
    sir = r["corpus_sir"]
    sir_t = r["corpus_sir_type"]
    sir_tr = r["corpus_sir_traj"]
    size = np.mean([r.get(f"avg_trie_size_hop_{s}", 0) for s in range(1, INDEX_LEN + 1)])
    n = r["num_questions"]
    print(f"{name:<28} {sir:<10.4f} {sir_t:<12.4f} {sir_tr:<12.4f} {size:<10.1f} {n:<8}")

In [ ]:
# @title 8. Results: Per-Hop Stratified SIR
print(f"\nPER-HOP DECOMPOSED SIR - §3.10.3\n")

for metric_name, metric_key in [("SIR* (combined)", "sir"),
                                 ("SIR*_type", "sir_type"),
                                 ("SIR*_traj", "sir_traj")]:
    print(f"--- {metric_name} ---")
    hdr = f"{'System':<28}"
    for s in range(1, INDEX_LEN + 1):
        hdr += f"{'Hop-' + str(s):<12}"
    print(hdr)
    print("-" * len(hdr))

    for name, r in all_results.items():
        if r is None:
            continue
        line = f"{name:<28}"
        for s in range(1, INDEX_LEN + 1):
            val = r.get(f"{metric_key}_hop_{s}", 0.0)
            line += f"{val:<12.4f}"
        print(line)
    print()

In [ ]:
# @title 9. SIR Reduction vs GCR
print("SIR REDUCTION vs GCR BASELINE\n")

if "GCR" in all_results and all_results["GCR"]:
    gcr_sir = all_results["GCR"]["corpus_sir"]
    gcr_sir_type = all_results["GCR"]["corpus_sir_type"]
    gcr_sir_traj = all_results["GCR"]["corpus_sir_traj"]

    for name, r in all_results.items():
        if name == "GCR" or r is None:
            continue
        red = (gcr_sir - r["corpus_sir"]) / max(gcr_sir, 1e-8) * 100
        red_t = (gcr_sir_type - r["corpus_sir_type"]) / max(gcr_sir_type, 1e-8) * 100
        red_tr = (gcr_sir_traj - r["corpus_sir_traj"]) / max(gcr_sir_traj, 1e-8) * 100
        print(f"{name}:")
        print(f"  SIR* reduction:      {red:.1f}%")
        print(f"  SIR*_type reduction:  {red_t:.1f}%")
        print(f"  SIR*_traj reduction:  {red_tr:.1f}%")
        print()

In [ ]:
# @title 10. Interpretation
print("""
=== INTERPRETATION (§3.4.4) ===

SIR*_type isolates the contribution of type blindness.
  - GCR: high (no type filtering)
  - DCA-Trie v1/v2: ~0 (hard type gate kills these)

SIR*_traj isolates trajectory blindness.
  - GCR: high (no semantic filtering)
  - DCA-Trie v1: medium (pre-filters against question semantics at τ)
  - DCA-Trie v2: lowest (residual vector tightens constraint set step by step)

SIR*_traj is expected to increase with hop depth for GCR and v1,
but stay flat or decrease for v2 because the residual vector narrows.

All three metrics are reported to attribute the contribution of each
filtering mechanism independently (§3.4.4, responding to Gap 3 in Ch. 2).
""")